# Temperature Sweep Study: Gas Swelling in Nuclear Fuel

This notebook demonstrates how to perform a **temperature sweep study** to investigate how temperature affects fission gas bubble evolution and swelling in nuclear fuel materials.

## Learning Objectives

By the end of this tutorial, you will:
- Understand the gas swelling model parameters
- Learn how to run simulations at different temperatures
- Analyze how temperature affects bubble formation and swelling
- Create publication-quality visualizations

## Physics Background

During nuclear fission:
1. **Gas atoms** (Xe, Kr) are produced and diffuse through the fuel
2. **Bubbles nucleate** when gas atoms cluster together
3. **Bubbles grow** by absorbing more gas and vacancies
4. **Swelling occurs** as bubbles occupy more volume

Temperature critically affects all these processes through:
- **Diffusion rates** (higher T → faster diffusion)
- **Gas pressure** in bubbles (affects growth rate)
- **Vacancy emission** from bubbles (thermal activation)
- **Bubble stability** (surface tension vs gas pressure)

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from gas_swelling.models.modelrk23 import GasSwellingModel
from gas_swelling.params.parameters import create_default_parameters

# Configure plotting for better visuals
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")

## 1. Understanding Default Parameters

Let's examine the default parameters and understand what they mean physically.

In [ ]:
# Create default parameters
params = create_default_parameters()

# Display key parameters with physical meanings
print("=" * 70)
print("DEFAULT MODEL PARAMETERS")
print("=" * 70)
print(f"\nTemperature: {params['temperature']} K")
print("  → Operating temperature of the fuel")
print(f"  → Typical range: 600-1000 K for metallic fuel")

print(f"\nFission rate: {params['fission_rate']:.2e} fissions/(m³·s)")
print("  → Rate of fission reactions producing gas atoms")
print("  → Higher = more gas production = faster swelling")

print(f"\nDislocation density: {params['dislocation_density']:.2e} m⁻²")
print("  → Density of crystal defects (dislocations)")
print("  → Acts as sinks for vacancies and interstitials")
print("  → Affects defect concentrations and bubble growth")

print(f"\nSurface energy: {params['surface_energy']} J/m²")
print("  → Energy cost of creating bubble surfaces")
print("  → Higher surface energy = smaller bubbles (minimizes surface area)")

print(f"\nBulk nucleation factor (Fnb): {params['Fnb']:.2e}")
print(f"Boundary nucleation factor (Fnf): {params['Fnf']:.2e}")
print("  → Control the rate of new bubble formation")
print("  → Lower values = fewer bubbles, each bubble grows larger")

print(f"\nGas equation of state: {params['eos_model']}")
print("  → 'ideal': Ideal gas law (PV = nRT)")
print("  → 'ronchi': Modified Van der Waals EOS for high-pressure gas")

## 2. Running a Single Simulation

Before doing a temperature sweep, let's run one simulation to understand the model output.

In [ ]:
# Create model with default parameters
model = GasSwellingModel(params)

# Simulation time: 100 days
sim_time_days = 100
sim_time_sec = sim_time_days * 24 * 3600

# Time points for output
t_eval = np.linspace(0, sim_time_sec, 100)

print(f"Running simulation for {sim_time_days} days at {params['temperature']} K...")
print("(This may take 10-30 seconds...)")

# Solve the system of ODEs
result = model.solve(
    t_span=(0, sim_time_sec),
    t_eval=t_eval
)

print("\nSimulation completed!")

In [ ]:
# Analyze the results
time_days = result['time'] / (24 * 3600)  # Convert to days

# Calculate swelling over time
V_bubbles_bulk = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
V_bubbles_boundary = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
swelling = (V_bubbles_bulk + V_bubbles_boundary) * 100  # Convert to percent

print("=" * 70)
print("SIMULATION RESULTS")
print("=" * 70)
print(f"\nFinal bubble radii:")
print(f"  Bulk bubbles: {result['Rcb'][-1]*1e9:.2f} nm")
print(f"  Boundary bubbles: {result['Rcf'][-1]*1e9:.2f} nm")

print(f"\nFinal bubble concentrations:")
print(f"  Bulk bubbles: {result['Ccb'][-1]:.2e} m⁻³")
print(f"  Boundary bubbles: {result['Ccf'][-1]:.2e} m⁻³")

print(f"\nFinal swelling: {swelling[-1]:.4f}%")

print(f"\nGas atoms per bubble:")
print(f"  Bulk bubbles: {result['Ncb'][-1]:.0f} atoms/bubble")
print(f"  Boundary bubbles: {result['Ncf'][-1]:.0f} atoms/bubble")

In [ ]:
# Visualize the results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Gas Swelling at {params["temperature"]} K', fontsize=14, fontweight='bold')

# Plot 1: Bubble radius evolution
axes[0, 0].plot(time_days, result['Rcb']*1e9, label='Bulk bubbles', linewidth=2)
axes[0, 0].plot(time_days, result['Rcf']*1e9, label='Boundary bubbles', linewidth=2)
axes[0, 0].set_xlabel('Time (days)')
axes[0, 0].set_ylabel('Bubble radius (nm)')
axes[0, 0].set_title('Bubble Radius Evolution')
axes[0, 0].legend()

# Plot 2: Swelling evolution
axes[0, 1].plot(time_days, swelling, color='red', linewidth=2)
axes[0, 1].set_xlabel('Time (days)')
axes[0, 1].set_ylabel('Swelling (%)')
axes[0, 1].set_title('Fuel Swelling')

# Plot 3: Bubble concentration
axes[1, 0].semilogy(time_days, result['Ccb'], label='Bulk', linewidth=2)
axes[1, 0].semilogy(time_days, result['Ccf'], label='Boundary', linewidth=2)
axes[1, 0].set_xlabel('Time (days)')
axes[1, 0].set_ylabel('Bubble concentration (m⁻³)')
axes[1, 0].set_title('Bubble Concentration')
axes[1, 0].legend()

# Plot 4: Gas atoms per bubble
axes[1, 1].plot(time_days, result['Ncb'], label='Bulk', linewidth=2)
axes[1, 1].plot(time_days, result['Ncf'], label='Boundary', linewidth=2)
axes[1, 1].set_xlabel('Time (days)')
axes[1, 1].set_ylabel('Gas atoms per bubble')
axes[1, 1].set_title('Gas Accumulation in Bubbles')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("Plots generated successfully!")

## 3. Temperature Sweep Study

Now let's investigate how temperature affects swelling by running simulations at multiple temperatures.

**Key Questions:**
- How does bubble size change with temperature?
- Is there an optimal temperature for maximum swelling?
- What mechanisms drive these trends?

In [ ]:
# Define temperature range for the sweep
# We'll explore temperatures from 600 K to 1000 K
temperatures = np.array([600, 650, 700, 750, 800, 850, 900, 950, 1000])

print("=" * 70)
print("TEMPERATURE SWEEP STUDY")
print("=" * 70)
print(f"\nRunning simulations at {len(temperatures)} temperatures...")
print(f"Temperature range: {temperatures[0]} - {temperatures[-1]} K")
print("\nThis may take 1-2 minutes...\n")

# Storage for results
results = {}

# Run simulation at each temperature
for i, temp in enumerate(temperatures):
    print(f"[{i+1}/{len(temperatures)}] Running at {temp} K...", end=' ')
    
    # Create parameters with this temperature
    params_temp = create_default_parameters()
    params_temp['temperature'] = temp
    
    # Create model and solve
    model_temp = GasSwellingModel(params_temp)
    result_temp = model_temp.solve(
        t_span=(0, sim_time_sec),
        t_eval=t_eval
    )
    
    # Calculate swelling
    Vb = (4/3) * np.pi * result_temp['Rcb']**3 * result_temp['Ccb']
    Vf = (4/3) * np.pi * result_temp['Rcf']**3 * result_temp['Ccf']
    swelling_temp = (Vb + Vf) * 100
    
    # Store results
    results[temp] = {
        'time': time_days,
        'swelling': swelling_temp,
        'Rcb': result_temp['Rcb'],
        'Rcf': result_temp['Rcf'],
        'Ccb': result_temp['Ccb'],
        'Ccf': result_temp['Ccf'],
        'Ncb': result_temp['Ncb'],
        'Ncf': result_temp['Ncf']
    }
    
    print(f"Done! Final swelling: {swelling_temp[-1]:.4f}%")

print("\n" + "=" * 70)
print("All simulations completed!")
print("=" * 70)

## 4. Analyzing Temperature Effects

Let's create comprehensive visualizations to understand the temperature dependence.

In [ ]:
# Extract final values at each temperature
final_swelling = []
final_Rcb = []
final_Rcf = []
final_Ccb = []
final_Ccf = []

for temp in temperatures:
    final_swelling.append(results[temp]['swelling'][-1])
    final_Rcb.append(results[temp]['Rcb'][-1] * 1e9)  # Convert to nm
    final_Rcf.append(results[temp]['Rcf'][-1] * 1e9)  # Convert to nm
    final_Ccb.append(results[temp]['Ccb'][-1])
    final_Ccf.append(results[temp]['Ccf'][-1])

final_swelling = np.array(final_swelling)
final_Rcb = np.array(final_Rcb)
final_Rcf = np.array(final_Rcf)

# Create temperature analysis plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Swelling vs Temperature
axes[0].plot(temperatures, final_swelling, 'o-', linewidth=2, markersize=8, color='red')
axes[0].set_xlabel('Temperature (K)')
axes[0].set_ylabel('Final Swelling (%)')
axes[0].set_title('Temperature Dependence of Swelling', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Highlight peak swelling
peak_idx = np.argmax(final_swelling)
axes[0].plot(temperatures[peak_idx], final_swelling[peak_idx], 'r*', 
            markersize=20, label=f'Peak: {temperatures[peak_idx]} K')
axes[0].legend()

# Plot 2: Bubble radius vs Temperature
axes[1].plot(temperatures, final_Rcb, 's-', linewidth=2, markersize=8, label='Bulk bubbles')
axes[1].plot(temperatures, final_Rcf, '^-', linewidth=2, markersize=8, label='Boundary bubbles')
axes[1].set_xlabel('Temperature (K)')
axes[1].set_ylabel('Final bubble radius (nm)')
axes[1].set_title('Temperature Dependence of Bubble Size', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Bubble concentration vs Temperature
axes[2].semilogy(temperatures, np.array(final_Ccb), 's-', linewidth=2, markersize=8, label='Bulk')
axes[2].semilogy(temperatures, np.array(final_Ccf), '^-', linewidth=2, markersize=8, label='Boundary')
axes[2].set_xlabel('Temperature (K)')
axes[2].set_ylabel('Final bubble concentration (m⁻³)')
axes[2].set_title('Temperature Dependence of Bubble Concentration', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Temperature analysis plots generated!")

In [ ]:
# Swelling evolution at different temperatures
fig, ax = plt.subplots(figsize=(12, 7))

# Define a colormap for temperatures
colors = plt.cm.viridis(np.linspace(0, 1, len(temperatures)))

# Plot swelling curves for each temperature
for i, temp in enumerate(temperatures):
    ax.plot(results[temp]['time'], results[temp]['swelling'],
            linewidth=2, color=colors[i], label=f'{temp} K')

ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Swelling (%)', fontsize=12)
ax.set_title('Swelling Evolution at Different Temperatures', fontsize=14, fontweight='bold')
ax.legend(ncol=3, loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Swelling evolution plot generated!")

## 5. Physical Interpretation

Let's interpret the observed trends based on physical mechanisms:

In [ ]:
# Find and print key observations
peak_idx = np.argmax(final_swelling)
peak_temp = temperatures[peak_idx]
peak_swelling = final_swelling[peak_idx]

print("=" * 70)
print("PHYSICAL INTERPRETATION")
print("=" * 70)

print(f"\n1. PEAK SWELLING")
print(f"   Maximum swelling: {peak_swelling:.4f}% at {peak_temp} K")
print(f"\n   Physical explanation:")
print(f"   - At low T (< 700 K): Slow diffusion limits gas transport to bubbles")
print(f"   - At intermediate T (700-850 K): Optimal balance of diffusion and growth")
print(f"   - At high T (> 900 K): Thermal vacancy emission reduces bubble stability")

print(f"\n2. BUBBLE SIZE TREND")
print(f"   Bulk bubbles grow from {final_Rcb[0]:.2f} nm at {temperatures[0]} K")
print(f"                      to {final_Rcb[-1]:.2f} nm at {temperatures[-1]} K")
print(f"\n   Physical explanation:")
print(f"   - Higher temperature → faster diffusion → larger bubbles")
print(f"   - Gas pressure increases with T, overcoming surface tension")

print(f"\n3. BUBBLE CONCENTRATION")
bulk_range = final_Ccb[-1] / final_Ccb[0]
print(f"   Bulk bubble concentration changes by factor of {bulk_range:.2e}")
print(f"\n   Physical explanation:")
print(f"   - Nucleation vs growth competition")
print(f"   - At low T: Many small bubbles nucleate")
print(f"   - At high T: Fewer, larger bubbles dominate")

print(f"\n4. SWELLING MECHANISM")
print(f"   Swelling = (bubble radius)³ × (bubble concentration)")
print(f"   Competition between:")
print(f"   - R³: Increases with T (larger bubbles)")
print(f"   - Cc: May decrease with T (fewer bubbles)")
print(f"   → Result: Non-monotonic temperature dependence")

print("=" * 70)

## 6. Interactive Exploration

Now try modifying parameters yourself! Here are some suggestions:

In [ ]:
# EXPERIMENT: Try different fission rates
print("EXPERIMENT: Effect of Fission Rate")
print("=" * 70)

fission_rates = [1e19, 3e19, 5e19, 7e19]  # fissions/(m³·s)
test_temp = 750  # K (near peak swelling)

print(f"\nTesting fission rates: {fission_rates}")
print(f"At constant temperature: {test_temp} K")
print("\nRunning simulations...\n")

fr_results = []
for fr in fission_rates:
    # Create parameters
    p = create_default_parameters()
    p['temperature'] = test_temp
    p['fission_rate'] = fr
    
    # Run simulation
    m = GasSwellingModel(p)
    r = m.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    # Calculate swelling
    Vb = (4/3) * np.pi * r['Rcb']**3 * r['Ccb']
    Vf = (4/3) * np.pi * r['Rcf']**3 * r['Ccf']
    s = (Vb + Vf) * 100
    
    fr_results.append(s[-1])
    print(f"  Fission rate {fr:.1e}: Final swelling = {s[-1]:.4f}%")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.array(fission_rates)/1e19, fr_results, 'o-', linewidth=2, markersize=10)
ax.set_xlabel('Fission Rate (×10¹⁹ fissions/(m³·s))')
ax.set_ylabel('Final Swelling (%)')
ax.set_title(f'Effect of Fission Rate at {test_temp} K')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservation: Higher fission rate → more gas production → more swelling")

## 7. Summary and Key Takeaways

### What We Learned

1. **Temperature has a non-monotonic effect on swelling**
   - Peak swelling occurs at intermediate temperatures (~750-800 K)
   - This is a well-known phenomenon in metallic fuels

2. **Competing mechanisms create the peak**
   - **Low T regime**: Diffusion-limited, slow gas transport
   - **Intermediate T regime**: Optimal for bubble growth
   - **High T regime**: Thermal effects reduce bubble stability

3. **Bubble size increases monotonically with T**
   - Faster diffusion → larger bubbles
   - Gas pressure overcomes surface tension more easily

4. **Bubble concentration can decrease with T**
   - Fewer nucleation events at higher T
   - Growth dominates over nucleation

### Further Exploration

Try these experiments on your own:
- **Vary fission rate**: How does gas production rate affect swelling?
- **Change dislocation density**: How do defect sinks influence results?
- **Modify nucleation factors**: What happens with more/ fewer bubbles?
- **Different simulation times**: Is the peak temperature time-dependent?
- **Compare U-Zr vs U-Pu-Zr**: Change composition parameters

### Next Steps

1. **Explore the code**:
   - Read `gas_swelling/models/modelrk23.py` for the rate equations
   - Check `gas_swelling/params/parameters.py` for all available parameters

2. **Run the quickstart tutorial**:
   ```bash
   python examples/quickstart_tutorial.py
   ```

3. **Read the documentation**:
   - `docs/parameter_reference.md`: Detailed parameter explanations
   - `CLAUDE.md`: Model physics and equations
   - `model_design.md`: Theoretical framework (in Chinese)

4. **Cite the model**:
   - Reference the original paper when using this model
   - See `original paper of swelling rate theory.md`

---

**Congratulations!** You've completed the temperature sweep tutorial. You now have the skills to:
- Run parametric studies with the gas swelling model
- Analyze temperature effects on bubble evolution
- Create publication-quality visualizations
- Design your own simulation experiments

Happy simulating! 🚀